# Comparative Analysis of Clustering Algorithms

This notebook compares centroid-, density-, hierarchical-, and probabilistic-clustering methods on the same synthetic dataset.

**Author:** Muhammad Kamil Shah

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering, DBSCAN, OPTICS, MeanShift
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score


## Prepare a common dataset

In [ ]:
X, _ = make_blobs(n_samples=600, centers=4, cluster_std=[0.7, 1.1, 0.8, 1.0], random_state=42)
X = StandardScaler().fit_transform(X)
plt.figure(figsize=(8, 5))
plt.scatter(X[:, 0], X[:, 1], s=14)
plt.title('Input data')
plt.show()

## Fit and evaluate the algorithms

In [ ]:
models = {
    'Agglomerative': AgglomerativeClustering(n_clusters=4),
    'DBSCAN': DBSCAN(eps=0.28, min_samples=8),
    'OPTICS': OPTICS(min_samples=8, xi=0.05),
    'Mean Shift': MeanShift(),
    'Gaussian Mixture': GaussianMixture(n_components=4, random_state=42),
}

results = []
labels_by_model = {}
for name, model in models.items():
    start = time.perf_counter()
    labels = model.fit_predict(X)
    elapsed = time.perf_counter() - start
    labels_by_model[name] = labels
    valid = labels != -1
    unique = set(labels[valid])
    score = silhouette_score(X[valid], labels[valid]) if len(unique) > 1 else np.nan
    results.append({
        'Algorithm': name,
        'Clusters': len(unique),
        'Noise points': int((labels == -1).sum()),
        'Silhouette score': score,
        'Runtime seconds': elapsed,
    })
pd.DataFrame(results).sort_values('Silhouette score', ascending=False)

## Visual comparison

In [ ]:
for name, labels in labels_by_model.items():
    plt.figure(figsize=(8, 5))
    plt.scatter(X[:, 0], X[:, 1], c=labels, s=14)
    plt.title(name)
    plt.show()

## Interpretation

Silhouette score summarizes cluster separation, while the number of noise points is especially important for DBSCAN and OPTICS. Runtime and the expected shape of clusters should also influence model choice. Density-based methods are useful when noise matters; hierarchical methods are interpretable; Gaussian mixtures support soft probabilistic structure.